# TD9: Communicative efficiency and color naming

**NB: YOU NEED TO DOWNLOAD WCS DATA FROM HERE: https://linguistics.berkeley.edu/wcs/data.html AFTER DOWNLOADING, PUT IT IN THE WCS FOLDER.**

## 0. Question

Today, we will use the **World Color Survey (WCS)** to look at communicative efficiency in the domain of color naming. In particular, we will ask three simple questions:

1. How does one language partition the color space?
2. How can we compute listener uncertainty, or surprisal, for each color chip?
3. Do languages with more color terms tend to have lower average surprisal?

This practical is loosely inspired by [Gibson et al. (2017)](https://www.pnas.org/doi/pdf/10.1073/pnas.1619666114) and [Zaslavsky et al. (2018)](https://www.pnas.org/doi/10.1073/pnas.1800521115). As usual, we will work step by step and keep the code as transparent as possible.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

sns.set(context='notebook', style='ticks',
        font_scale=1.2, palette='colorblind')


## 1. Data

In the `WCS` folder, you will find the raw files from the World Color Survey. For the main analysis, we only need three of them:

- `term.txt`: naming responses by language, speaker, and chip
- `chip.txt`: the position of each chip in the WCS grid
- `cielab.txt`: perceptual coordinates for each chip in CIELAB space

We will also load `langs_info.txt`, so that we can keep track of language names later on.


In [ ]:
BASE = 'WCS'

responses = pd.read_csv(
    f'{BASE}/term.txt',
    sep=r'\s+',
    header=None,
    names=['language_id', 'speaker_id', 'chip_id', 'response_code']
)

chips = pd.read_csv(
    f'{BASE}/chip.txt',
    sep=r'\s+',
    header=None,
    names=['chip_id', 'row', 'col', 'grid_id']
)

lab = pd.read_csv(
    f'{BASE}/cielab.txt',
    sep=r'\s+',
    header=None,
    comment='#',
    names=['chip_id', 'V', 'H', 'C', 'MunH', 'MunV', 'L_star', 'a_star', 'b_star']
)

langs = pd.read_csv(
    f'{BASE}/langs_info.txt',
    sep='\t',
    header=None,
    usecols=[0, 1],
    names=['language_id', 'language']
)


Let's look at the naming responses first. Each row corresponds to one naming response for one chip by one speaker.


In [ ]:
responses.head(10)


Now let's look at the chip table. This gives us the position of each chip in the WCS grid.


In [ ]:
chips.head(10)


Finally, `cielab.txt` gives the perceptual coordinates of each chip in the CIELAB color space. We will mainly use `L_star`, `a_star`, and `b_star`.


In [ ]:
lab.head(10)

Now let's combine the response table with the chip information and the CIELAB coordinates. You should merge the tables on the `chip_id` column, and keep the relevant columns from each table (`L_star`, `a_star`, `b_star` from lab, and everything from chips and lab. You can also add language from `langs`, by joining by `language_id`.


In [ ]:
# Merge the response table with the CIELAB, chip, and language tables.
# Store the result back in `responses`.
##################
# YOUR CODE HERE #
##################

For plotting, it is convenient to convert the row column (`A`, `B`, `C`, ...) into numbers into reverse order.


In [ ]:
responses['row_num'] = responses['row'].map(lambda x: ord('J') - ord(x))
responses.head(10)

Let's inspect one language first. What do you notice about how many responses each chip can receive?


In [ ]:
responses.query('language_id == 1').groupby('chip_id')['response_code'].value_counts().head(20)

## 2. From Responses to a Partition

Each chip can receive several naming responses from different speakers. To get a simple partition of the color space for one language, we can assign each chip its **most frequent** response code. Do it using `groupby` and `agg`, combined with a lambda function that computes the most frequent response code for each chip. You can also keep the row and column information for each chip, which will be useful for plotting later on.


In [ ]:
language_id = 1

# Build `language_partition` dataframe with one row per chip and the most frequent
# response code for that chip. Keep the plotting coordinates too.
##################
# YOUR CODE HERE #
##################

Now plot this partition for language 10. Each square corresponds to one chip, and the color of the square shows the most frequent response code for that chip. Use seaborn's `scatteplot' function, and square markers to plot the color map.


In [ ]:
plt.figure(figsize=(12, 3.6))
sns.scatterplot(
    data=language_partition,
    x='col', y='row_num',
    hue='response_code',
    palette='Paired',
    s=200,
    marker='s'
)
sns.despine(left=True, bottom=True)
plt.xticks([])
plt.yticks([])
plt.xlabel('')
plt.ylabel('')
plt.legend(title='Response code', 
           bbox_to_anchor=(0.97, 0.94), loc='upper left')
plt.show()

## 3. Surprisal of Each Chip

Now we will compute listener uncertainty for each chip. We will use the following definition:

$$S(c)=\sum_w P(w\mid c)\log_2 \frac{1}{P(c\mid w)}$$

where

$$P(c\mid w)=\frac{P(w\mid c)P(c)}{P(w)}$$

Assume a uniform prior over chips within a language, so that $P(c)=1/N_c$, where $N_c$ is the number of chips.

We will first compute everything for one language and one chip, and then turn the procedure into a function. Take chip 1 in language 1 as an example. Compute $P(w\mid c)$ for each response code $w$ (so what is the probability of each word to be used with this chip, use `value_counts(normalize=True)' for this), and then compute $P(c\mid w)$ (probability of using each word across all chips) and then you can surprisal $S(c)$ for this chip.


In [ ]:
language_id = 1
language_responses = responses.query('language_id == @language_id').copy()
chip_id = 1

Let's compute $P(w\mid c)$ for each word $w$ for chip 1 in language 10:

In [ ]:
# Compute the distribution of response codes for this chip and language, 
# and store it in `chip_word_probs`. 
# Normalize the distribution so that it sums to 1 
# (use `normalize=True` in `value_counts`).
##################
# YOUR CODE HERE #
##################

This gives us $P(w\mid c)$ for chip 1 in language 1. Now compute the overall probability of each word in this language, that is, $P(w)$. Use the same approach, but this time don't filter by chip id.


In [74]:
# Compute the overall probability of each response code in this language
# and store it in `word_probs`.
##################
# YOUR CODE HERE #
##################

Now we have everything we need to compute one surprisal contribution. Let's do it for one word first.


In [75]:
example_word = chip_word_probs.index[0]
p_c = 1 / 220
##################
# YOUR CODE HERE #
##################

Now let's write a small function that computes surprisal for one chip. The function takes the responses of one language and one chip ID as input.


In [76]:
def chip_surprisal(language_responses, chip_id):
    surprisal = 0
    ##################
    # YOUR CODE HERE #
    ##################
    return surprisal

Let's test the function on chip 1 in language 1.


In [77]:
chip_surprisal(language_responses, 1)

0

Now convert this into a function that computes the **average** surprisal across all chips in one language. Use the `chip_surisal' function to compute this.


In [ ]:
def language_surprisal(language_id, chips_subset=None):
    ##################
    # YOUR CODE HERE #
    ##################

    pass

Tesit it for language 1.

In [ ]:
language_surprisal(1)

Now let's plot a color map for a language with surpisal instead of categories:

In [ ]:
# compute surprisal for all chips of one language, and save it together
# with the chip coordinates for plotting

##################
# YOUR CODE HERE #
##################
# Create a dataframe called `language_responses` with one row per chip
# and a `surprisal` column.

In [ ]:
plot_data = language_responses

fig, ax = plt.subplots(figsize=(12, 3.6))
sc = ax.scatter(
    plot_data['col'],
    plot_data['row_num'],
    c=plot_data['surprisal'],
    cmap='viridis_r',
    s=100,
    marker='s'
)

ax.invert_yaxis()
sns.despine(left=True, bottom=True)
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')

fig.colorbar(sc, ax=ax, label='Surprisal')
plt.show()

Now apply this function to all languages and store the results in a dataframe together with the number of distinct response codes in each language.


In [78]:
# Build `language_scores` with one row per language, together with
# `average_surprisal`, `num_words`, and the language metadata.
##################
# YOUR CODE HERE #
##################

## 4. Compare Languages

Now let's see whether languages with more color terms tend to have lower average surprisal. First, let's compute the log number of response codes, since the number of response codes varies widely across languages.


In [80]:
# Add a `log_num_words` column to `language_scores`.
##################
# YOUR CODE HERE #
##################

In [ ]:
plt.figure(figsize=(8, 4))
sns.scatterplot(data=language_scores, x='log_num_words', y='average_surprisal')
sns.regplot(data=language_scores, x='log_num_words', y='average_surprisal', scatter=False, color='red')
plt.xlabel('Number of color words (log scale)')
plt.ylabel('Average surprisal')
sns.despine()
plt.show()


Compute the correlation between log number of response codes and average surprisal across languages. Use the Spearman correlation -- `stats.spearmanr`.

In [81]:
# Compute the Spearman correlation between `log_num_words` and
# `average_surprisal`.
##################
# YOUR CODE HERE #
##################

What does this analysis suggest? Is the relationship clearly linear, or do you see a lot of variation among languages with similar numbers of terms?


## 5. Additional Analysis

As a simple extension, let's compare darker and lighter chips. We will split the chips into two groups using the median value of `L_star`. Then we will compute the average surprisal for each group in each language.


In [ ]:
median_l = lab['L_star'].median()
dark_chips = set(lab.loc[lab['L_star'] < median_l, 'chip_id'])
light_chips = set(lab.loc[lab['L_star'] >= median_l, 'chip_id'])

light_dark_scores = []

# Compute surprisal scores for both subsets of chips. Append a tuple
# (language_id, type (light/dark), average_surprisal) to `light_dark_scores`.
##################
# YOUR CODE HERE #
##################

light_dark_scores = pd.DataFrame(light_dark_scores, columns=['language_id', 'chip_type', 'average_surprisal'])
light_dark_scores = light_dark_scores.merge(langs, on='language_id', how='left')
light_dark_scores.head()

,language_id,chip_type,average_surprisal,language


Now let's plot this. Use `sns.pointplot` and `hue='language_id'` to match languages by light/dark surpisal.

In [ ]:
plt.figure(figsize=(8, 6))

sns.pointplot(
    data=light_dark_scores,
    x='chip_type',
    y='average_surprisal',
    hue='language_id',
    legend=False,
)

plt.xlabel('')
plt.ylabel('Average surprisal')
sns.despine()
plt.show()

Now let's do a paired t-test to see whether the difference between light and dark chips is significant across languages. Don't forget to sort by language ID before doing the test, so that the order of languages is the same in both groups. Use `stats.ttest_rel` for this.

In [ ]:
from scipy.stats import ttest_rel

# Run a paired t-test comparing dark-chip and light-chip surprisal.
##################
# YOUR CODE HERE #
##################


What does this analysis suggest? Do languages tend to have higher surprisal for light or dark chips? Is the difference significant across languages?